# K-means TC4 Fashion-MNIST: tái chạy trên Colab

Notebook độc lập để chạy lại duy nhất bước fit codebook TC4. Không cần clone repo hay tải file từ Google Drive. Notebook dùng Python kernel mặc định của Colab, cài Julia 1.13.3 và khóa Clustering.jl 0.15.8 cùng dependency theo Manifest release.

**Chọn runtime CPU** ở Runtime → Change runtime type. Toàn bộ 70.000 ảnh, gồm cả split test, được dùng tạo descriptor và fit K-means. Nhãn chỉ được kiểm tra tính toàn vẹn, không đi vào descriptor hay quá trình fit.

K-means báo số **iterations** (vòng lặp Lloyd), không phải epoch như mạng nơ-ron. Cấu hình release giới hạn 300 vòng, nên kết quả 372 không thể đến từ đúng cấu hình này. Kết quả release đã lưu trong repo là 139 vòng; notebook đo lượt Colab mới và không ép kết quả phải bằng 139.


## 1. Cấu hình release

Patch 4×4 không chồng lấp; giữ patch nếu có pixel >29; descriptor raw pixel 16 chiều, Float32/255 và column-major trong patch; không sampling; K=32; seed=123; K-means++; squared Euclidean; tối đa 300 vòng; tolerance 1e-4.


In [ ]:
from pathlib import Path
import os, json, hashlib

CONFIG = {
    "dataset": "Fashion-MNIST", "train_images": 60_000, "test_images": 10_000,
    "all_images": 70_000, "patch_size": 4, "foreground_threshold": 29,
    "descriptor": "raw pixels Float32 divided by 255", "dimension": 16,
    "flatten_order": "column-major inside each patch",
    "fit_scope": "all retained descriptors from 70,000 images",
    "image_sampling": False, "descriptor_sampling": False,
    "k": 32, "seed": 123, "initialization": "kmeans++",
    "distance": "squared_euclidean", "max_iterations": 300,
    "tolerance": 1e-4, "julia_version": "1.13.3",
    "clustering_version": "0.15.8", "backend": "CPU",
    "expected_descriptors": 2_143_746,
}
WORKDIR = Path(os.environ.get(
    "FASHION_TC4_WORKDIR", "/content/fashion_tc4_kmeans_reproduction"
))
DATA_DIR = Path(os.environ.get("FASHION_TC4_DATA_DIR", str(WORKDIR / "data")))
OUTPUT_DIR = WORKDIR / "output"
for directory in (WORKDIR, DATA_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = OUTPUT_DIR / "config_snapshot.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + "\n")
print("Work directory:", WORKDIR)
print("Locked config SHA-256:", hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest())


## 2. Cài Julia và dependency đã khóa

Colab dùng Python kernel. Ô này tải Julia 1.13.3 Linux x86_64 từ máy chủ chính thức, kiểm tra SHA-256, rồi dựng môi trường Julia tối giản với Clustering.jl 0.15.8 và đúng dependency trong Manifest.


In [ ]:
import os, shutil, subprocess, urllib.request

JULIA_VERSION = CONFIG["julia_version"]
JULIA_URL = (
    "https://julialang-s3.julialang.org/bin/linux/x64/1.13/"
    "julia-1.13.3-linux-x86_64.tar.gz"
)
JULIA_SHA256 = "5c61cb4b40332cb5a3504e7779e2f5dac093caa4a1116ded19f01413f85c63e4"
JULIA_ENV_DIR = WORKDIR / "julia_env"
JULIA_ENV_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_TOML = "name = \"FashionKMeansColab\"\nuuid = \"724d92f3-0e82-42c7-97ad-e4db1d610ab0\"\nversion = \"0.1.0\"\n\n[deps]\nClustering = \"aaaa29a8-35af-508c-8bc3-b662a17a0fe5\"\nDistances = \"b4f34e82-e78d-54a5-968a-f98e89d6e8f7\"\nRandom = \"9a3f8284-a2c9-5f02-9a11-845980a1fd5c\"\nSHA = \"ea8e919c-243c-51af-8825-aaa63cd721ce\"\nStatistics = \"10745b16-79ce-11e8-11f9-7d13ad32a3b2\"\nTOML = \"fa267f1f-6049-4f14-aa54-33bafae1ed76\"\n\n[compat]\nClustering = \"=0.15.8\"\njulia = \"=1.13.3\""
MANIFEST_TOML = "julia_version = \"1.13.3\"\nmanifest_format = \"2.1\"\n\n[[deps.AbstractTrees]]\ngit-tree-sha1 = \"2d9c9a55f9c93e8887ad391fbae72f8ef55e1177\"\nregistries = \"General\"\nuuid = \"1520ce14-60c1-5f80-bbc7-55ef81b5835c\"\nversion = \"0.4.5\"\n\n[[deps.AliasTables]]\ndeps = [\"PtrArrays\", \"Random\"]\ngit-tree-sha1 = \"9876e1e164b144ca45e9e3198d0b689cadfed9ff\"\nregistries = \"General\"\nuuid = \"66dad0bd-aa9a-41b7-9441-69ab47430ed8\"\nversion = \"1.1.3\"\n\n[[deps.Artifacts]]\nuuid = \"56f22d72-fd6d-98f1-02f0-08ddc0907c33\"\nversion = \"1.11.0\"\n\n[[deps.Clustering]]\ndeps = [\"Distances\", \"LinearAlgebra\", \"NearestNeighbors\", \"Printf\", \"Random\", \"SparseArrays\", \"Statistics\", \"StatsBase\"]\ngit-tree-sha1 = \"3e22db924e2945282e70c33b75d4dde8bfa44c94\"\nregistries = \"General\"\nuuid = \"aaaa29a8-35af-508c-8bc3-b662a17a0fe5\"\nversion = \"0.15.8\"\n\n[[deps.CompilerSupportLibraries_jll]]\ndeps = [\"Artifacts\", \"Libdl\"]\nuuid = \"e66e0078-7015-5450-92f7-15fbd957f2ae\"\nversion = \"1.5.5+2\"\n\n[[deps.DataAPI]]\ngit-tree-sha1 = \"abe83f3a2f1b857aac70ef8b269080af17764bbe\"\nregistries = \"General\"\nuuid = \"9a962f9c-6df0-11e9-0e5d-c546b8b5ee8a\"\nversion = \"1.16.0\"\n\n[[deps.DataStructures]]\ndeps = [\"OrderedCollections\"]\ngit-tree-sha1 = \"b0bc6d2cad1fed8b7fd59a1551a991cb3d2809e6\"\nregistries = \"General\"\nuuid = \"864edb3b-99cc-5e75-8d2d-829cb0a9cfe8\"\nversion = \"0.19.6\"\n\n[[deps.Dates]]\ndeps = [\"Printf\"]\nuuid = \"ade2ca70-3891-5945-98fb-dc099432e06a\"\nversion = \"1.11.0\"\n\n[[deps.Distances]]\ndeps = [\"LinearAlgebra\", \"Statistics\", \"StatsAPI\"]\ngit-tree-sha1 = \"c7e3a542b999843086e2f29dac96a618c105be1d\"\nregistries = \"General\"\nuuid = \"b4f34e82-e78d-54a5-968a-f98e89d6e8f7\"\nversion = \"0.10.12\"\n\n    [deps.Distances.extensions]\n    DistancesChainRulesCoreExt = \"ChainRulesCore\"\n    DistancesSparseArraysExt = \"SparseArrays\"\n\n    [deps.Distances.weakdeps]\n    ChainRulesCore = \"d360d2e6-b24c-11e9-a2a3-2a2ae2dbcce4\"\n    SparseArrays = \"2f01184e-e22b-5df5-ae63-d93ebab69eaf\"\n\n[[deps.DocStringExtensions]]\ngit-tree-sha1 = \"38ef52f18cbdfa783ec6c25acc23fb1e8525ec16\"\nregistries = \"General\"\nuuid = \"ffbed154-4ef7-542d-bbb7-c09d3a79fcae\"\nversion = \"0.10.2\"\n\n[[deps.IrrationalConstants]]\ngit-tree-sha1 = \"d22e8bdb3f03c300a103ff23c0dd5a654c6701de\"\nregistries = \"General\"\nuuid = \"92d709cd-6900-40b7-9082-c6be49f344b6\"\nversion = \"0.2.7\"\n\n[[deps.Libdl]]\nuuid = \"8f399da3-3557-5675-b5ff-fb832c97cbdb\"\nversion = \"1.11.0\"\n\n[[deps.LinearAlgebra]]\ndeps = [\"Libdl\", \"OpenBLAS_jll\", \"libblastrampoline_jll\"]\nuuid = \"37e2e46d-f89d-539d-b4ee-838fcccc9c8e\"\nversion = \"1.13.0\"\n\n[[deps.LogExpFunctions]]\ndeps = [\"DocStringExtensions\", \"IrrationalConstants\", \"LinearAlgebra\"]\ngit-tree-sha1 = \"52847246e86dd3dd88c39716748c57fd2bc43dce\"\nregistries = \"General\"\nuuid = \"2ab3a3ac-af41-5b50-aa03-7779005ae688\"\nversion = \"1.0.6\"\n\n    [deps.LogExpFunctions.extensions]\n    LogExpFunctionsChainRulesCoreExt = \"ChainRulesCore\"\n    LogExpFunctionsChangesOfVariablesExt = \"ChangesOfVariables\"\n    LogExpFunctionsInverseFunctionsExt = \"InverseFunctions\"\n\n    [deps.LogExpFunctions.weakdeps]\n    ChainRulesCore = \"d360d2e6-b24c-11e9-a2a3-2a2ae2dbcce4\"\n    ChangesOfVariables = \"9e997f8a-9a97-42d5-a9f1-ce6bfc15e2c0\"\n    InverseFunctions = \"3587e190-3f89-42d0-90ee-14403ec27112\"\n\n[[deps.Missings]]\ndeps = [\"DataAPI\"]\ngit-tree-sha1 = \"d9d20d6eb2a3290ba5c70a1c013b45305cc39c9e\"\nregistries = \"General\"\nuuid = \"e1d29d7a-bbdc-5cf2-9ac0-f12de2c33e28\"\nversion = \"1.2.1\"\n\n[[deps.NearestNeighbors]]\ndeps = [\"AbstractTrees\", \"Distances\", \"StaticArrays\"]\ngit-tree-sha1 = \"576eb4656529c12e77a46b17c23103dfba9fa570\"\nregistries = \"General\"\nuuid = \"b8a86587-4115-5ab1-83bc-aa920d37bbce\"\nversion = \"0.4.29\"\n\n[[deps.OpenBLAS_jll]]\ndeps = [\"Artifacts\", \"CompilerSupportLibraries_jll\", \"Libdl\"]\nuuid = \"4536629a-c528-5b80-bd46-f80d51c5b363\"\nversion = \"0.3.30+0\"\n\n[[deps.OrderedCollections]]\ngit-tree-sha1 = \"b079a6affebde167dad8ffe0d670b53917fbcb68\"\nregistries = \"General\"\nuuid = \"bac558e1-5e72-5ebc-8fee-abe8a469f55d\"\nversion = \"2.0.3\"\n\n[[deps.PrecompileTools]]\ndeps = [\"Preferences\"]\ngit-tree-sha1 = \"1e697ad6f68af89cd3651bbe9b4a7b5d800adf5a\"\nregistries = \"General\"\nuuid = \"aea7be01-6a6a-4083-8856-8a6e6704d82a\"\nversion = \"1.4.0\"\n\n[[deps.Preferences]]\ndeps = [\"TOML\"]\ngit-tree-sha1 = \"aef1987d90dcc2e2cbd919ad10b2d25bcce4f418\"\nregistries = \"General\"\nuuid = \"21216c6a-2e73-6563-6e65-726566657250\"\nversion = \"1.6.1\"\n\n[[deps.Printf]]\ndeps = [\"Unicode\"]\nuuid = \"de0858da-6303-5e67-8744-51eddeeeb8d7\"\nversion = \"1.11.0\"\n\n[[deps.PtrArrays]]\ngit-tree-sha1 = \"4fbbafbc6251b883f4d2705356f3641f3652a7fe\"\nregistries = \"General\"\nuuid = \"43287f4e-b6f4-7ad1-bb20-aadabca52c3d\"\nversion = \"1.4.0\"\n\n[[deps.Random]]\ndeps = [\"SHA\"]\nuuid = \"9a3f8284-a2c9-5f02-9a11-845980a1fd5c\"\nversion = \"1.11.0\"\n\n[[deps.SHA]]\nuuid = \"ea8e919c-243c-51af-8825-aaa63cd721ce\"\nversion = \"1.0.0\"\n\n[[deps.Serialization]]\nuuid = \"9e88b42a-f829-5b0c-bbe9-9e923198166b\"\nversion = \"1.11.0\"\n\n[[deps.SortingAlgorithms]]\ndeps = [\"DataStructures\"]\ngit-tree-sha1 = \"13cd91cc9be159e3f4d95b857fa2aa383b53772a\"\nregistries = \"General\"\nuuid = \"a2af1166-a08f-5f64-846c-94a0d3cef48c\"\nversion = \"1.2.3\"\n\n[[deps.SparseArrays]]\ndeps = [\"Libdl\", \"LinearAlgebra\", \"Random\", \"Serialization\", \"SuiteSparse_jll\"]\nuuid = \"2f01184e-e22b-5df5-ae63-d93ebab69eaf\"\nversion = \"1.13.0\"\n\n[[deps.StaticArrays]]\ndeps = [\"LinearAlgebra\", \"PrecompileTools\", \"Random\", \"StaticArraysCore\"]\ngit-tree-sha1 = \"8a8a52f887f38d44014e24ee46b8a7311c625f6d\"\nregistries = \"General\"\nuuid = \"90137ffa-7385-5640-81b9-e52037218182\"\nversion = \"1.9.24\"\n\n    [deps.StaticArrays.extensions]\n    StaticArraysChainRulesCoreExt = \"ChainRulesCore\"\n    StaticArraysStatisticsExt = \"Statistics\"\n\n    [deps.StaticArrays.weakdeps]\n    ChainRulesCore = \"d360d2e6-b24c-11e9-a2a3-2a2ae2dbcce4\"\n    Statistics = \"10745b16-79ce-11e8-11f9-7d13ad32a3b2\"\n\n[[deps.StaticArraysCore]]\ngit-tree-sha1 = \"bf7e2d0d9ed9ac2a652fd598c0f784910cc759f2\"\nregistries = \"General\"\nuuid = \"1e83bf80-4336-4d27-bf5d-d5a4f845583c\"\nversion = \"1.4.5\"\n\n[[deps.Statistics]]\ndeps = [\"LinearAlgebra\"]\ngit-tree-sha1 = \"e2b53ce13a53367e96601081e33d34746b571bad\"\nregistries = \"General\"\nuuid = \"10745b16-79ce-11e8-11f9-7d13ad32a3b2\"\nversion = \"1.11.5\"\nweakdeps = [\"SparseArrays\"]\n\n    [deps.Statistics.extensions]\n    SparseArraysExt = [\"SparseArrays\"]\n\n[[deps.StatsAPI]]\ndeps = [\"LinearAlgebra\"]\ngit-tree-sha1 = \"178ed29fd5b2a2cfc3bd31c13375ae925623ff36\"\nregistries = \"General\"\nuuid = \"82ae8749-77ed-4fe6-ae5f-f523153014b0\"\nversion = \"1.8.0\"\n\n[[deps.StatsBase]]\ndeps = [\"AliasTables\", \"DataAPI\", \"DataStructures\", \"IrrationalConstants\", \"LinearAlgebra\", \"LogExpFunctions\", \"Missings\", \"Printf\", \"Random\", \"SortingAlgorithms\", \"SparseArrays\", \"Statistics\", \"StatsAPI\"]\ngit-tree-sha1 = \"adb9da019510162e67a4493fc235c23203d8b09e\"\nregistries = \"General\"\nuuid = \"2913bbd2-ae8a-5f71-8c99-4fb6c76f3a91\"\nversion = \"0.34.13\"\n\n[[deps.SuiteSparse_jll]]\ndeps = [\"Artifacts\", \"CompilerSupportLibraries_jll\", \"Libdl\", \"libblastrampoline_jll\"]\nuuid = \"bea87d4a-7f5b-5778-9afe-8cc45184846c\"\nversion = \"7.10.1+0\"\n\n[[deps.TOML]]\ndeps = [\"Dates\"]\nuuid = \"fa267f1f-6049-4f14-aa54-33bafae1ed76\"\nversion = \"1.0.3\"\n\n[[deps.Unicode]]\nuuid = \"4ec0a83e-493e-50e2-b9ac-8f72acf5a8f5\"\nversion = \"1.11.0\"\n\n[[deps.libblastrampoline_jll]]\ndeps = [\"Artifacts\", \"Libdl\"]\nuuid = \"8e850b90-86db-534c-a0d3-1478176c7d93\"\nversion = \"5.15.0+0\""
(JULIA_ENV_DIR / "Project.toml").write_text(PROJECT_TOML + "\n")
(JULIA_ENV_DIR / "Manifest.toml").write_text(MANIFEST_TOML + "\n")

def find_exact_julia():
    candidate = shutil.which("julia")
    if not candidate:
        return None
    version = subprocess.run(
        [candidate, "--version"], check=True, capture_output=True, text=True
    ).stdout.strip()
    return candidate if version.endswith(JULIA_VERSION) else None

JULIA_BIN = find_exact_julia()
if JULIA_BIN is None:
    archive = WORKDIR / "julia-1.13.3-linux-x86_64.tar.gz"
    have_hash = (
        hashlib.sha256(archive.read_bytes()).hexdigest()
        if archive.exists() else ""
    )
    if have_hash != JULIA_SHA256:
        temporary = archive.with_suffix(archive.suffix + ".download")
        with urllib.request.urlopen(JULIA_URL, timeout=180) as response:
            with temporary.open("wb") as target:
                shutil.copyfileobj(response, target)
        observed = hashlib.sha256(temporary.read_bytes()).hexdigest()
        if observed != JULIA_SHA256:
            temporary.unlink(missing_ok=True)
            raise ValueError(f"Julia archive SHA-256 mismatch: {observed}")
        temporary.replace(archive)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(WORKDIR)], check=True)
    JULIA_BIN = str(WORKDIR / "julia-1.13.3" / "bin" / "julia")

version = subprocess.run(
    [JULIA_BIN, "--version"], check=True, capture_output=True, text=True
).stdout.strip()
if not version.endswith(JULIA_VERSION):
    raise RuntimeError(f"Expected Julia {JULIA_VERSION}, got {version}")

JULIA_ENVIRONMENT = os.environ.copy()
JULIA_ENVIRONMENT.update({
    "JULIA_DEPOT_PATH": str(WORKDIR / "julia_depot"),
    "JULIA_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "JULIA_PKG_PRECOMPILE_AUTO": "0",
})
Path(JULIA_ENVIRONMENT["JULIA_DEPOT_PATH"]).mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        JULIA_BIN, "--startup-file=no", f"--project={JULIA_ENV_DIR}", "-e",
        "using Pkg; Pkg.instantiate(); Pkg.precompile(); "
        "using Clustering, Distances; "
        'println("Julia ", VERSION, " | Clustering.jl ", pkgversion(Clustering))',
    ],
    cwd=WORKDIR, env=JULIA_ENVIRONMENT, check=True,
)
print("Pinned Julia environment is ready.")


## 3. Tải và xác minh Fashion-MNIST

Bốn archive IDX gốc được kiểm tra bằng SHA-256. Notebook xác nhận header, kích thước, số ảnh và miền nhãn. Nhãn không được dùng để tạo descriptor.


In [ ]:
import gzip, shutil, struct, urllib.request
import numpy as np

FASHION_URL = (
    "https://raw.githubusercontent.com/zalandoresearch/"
    "fashion-mnist/master/data/fashion"
)
ARCHIVES = {
    "train-images-idx3-ubyte.gz": ("3aede38d61863908ad78613f6a32ed271626dd12800ba2636569512369268a84", "images", 60_000),
    "train-labels-idx1-ubyte.gz": ("a04f17134ac03560a47e3764e11b92fc97de4d1bfaf8ba1a3aa29af54cc90845", "labels", 60_000),
    "t10k-images-idx3-ubyte.gz": ("346e55b948d973a97e58d2351dde16a484bd415d4595297633bb08f03db6a073", "images", 10_000),
    "t10k-labels-idx1-ubyte.gz": ("67da17c76eaffca5446c3361aaab5c3cd6d1c2608764d35dfb1850b086bf8dd5", "labels", 10_000),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

archive_hashes = {}
for filename, (expected, _kind, _count) in ARCHIVES.items():
    path = DATA_DIR / filename
    if not path.exists() or sha256_file(path) != expected:
        temporary = path.with_suffix(path.suffix + ".download")
        with urllib.request.urlopen(f"{FASHION_URL}/{filename}", timeout=180) as response:
            with temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
        if sha256_file(temporary) != expected:
            temporary.unlink(missing_ok=True)
            raise ValueError(f"SHA-256 mismatch: {filename}")
        temporary.replace(path)
    archive_hashes[filename] = sha256_file(path)

def read_images(path, expected_count):
    with gzip.open(path, "rb") as source:
        payload = source.read()
    magic, count, rows, columns = struct.unpack(">IIII", payload[:16])
    if (magic, count, rows, columns) != (2051, expected_count, 28, 28):
        raise ValueError(f"Bad IDX image header: {path.name}")
    if len(payload) != 16 + count * rows * columns:
        raise ValueError(f"Bad IDX image payload: {path.name}")
    return np.frombuffer(payload, dtype=np.uint8, offset=16).reshape(
        count, rows, columns
    ).copy()

def read_labels(path, expected_count):
    with gzip.open(path, "rb") as source:
        payload = source.read()
    magic, count = struct.unpack(">II", payload[:8])
    if (magic, count) != (2049, expected_count) or len(payload) != 8 + count:
        raise ValueError(f"Bad IDX label file: {path.name}")
    labels = np.frombuffer(payload, dtype=np.uint8, offset=8)
    if np.any(labels > 9):
        raise ValueError(f"Label outside 0..9: {path.name}")
    return labels

train_images = read_images(DATA_DIR / "train-images-idx3-ubyte.gz", 60_000)
test_images = read_images(DATA_DIR / "t10k-images-idx3-ubyte.gz", 10_000)
train_labels = read_labels(DATA_DIR / "train-labels-idx1-ubyte.gz", 60_000)
test_labels = read_labels(DATA_DIR / "t10k-labels-idx1-ubyte.gz", 10_000)
assert set(np.unique(train_labels)) == set(range(10))
assert set(np.unique(test_labels)) == set(range(10))
images = np.concatenate((train_images, test_images), axis=0)
assert images.shape == (70_000, 28, 28) and images.dtype == np.uint8
del train_images, test_images, train_labels, test_labels

INPUT_MANIFEST = OUTPUT_DIR / "input_manifest.toml"
lines = [
    'dataset = "Fashion-MNIST"',
    "image_count = 70000",
    "train_count = 60000",
    "test_count = 10000",
    "",
    "[archive_sha256]",
]
lines += [f'"{name}" = "{digest}"' for name, digest in archive_hashes.items()]
INPUT_MANIFEST.write_text("\n".join(lines) + "\n")
print("Verified 4 dataset archives. Images loaded:", images.shape)
print("Labels checked only for archive integrity.")


## 4. Tạo lại descriptor raw pixel

Thứ tự duyệt image → patch-row → patch-column khớp TC1–TC3. Giữ patch nếu có pixel >29; vector 16 chiều dùng Float32/255 và column-major trong patch. Ghi descriptor theo từng khối để giảm RAM.


In [ ]:
import gc, time

DESCRIPTOR_BIN = WORKDIR / "retained_descriptors_float32.bin"
PATCH = CONFIG["patch_size"]
GRID = 28 // PATCH
CHUNK_IMAGES = 1_024
THRESHOLD = CONFIG["foreground_threshold"]

sample = images[:1].reshape(1, GRID, PATCH, GRID, PATCH).transpose(0, 1, 3, 2, 4)
for patch_row, patch_col in ((0, 0), (0, 1), (1, 0), (6, 6)):
    observed = sample[0, patch_row, patch_col].swapaxes(-1, -2).reshape(16)
    expected = images[
        0,
        patch_row * PATCH:(patch_row + 1) * PATCH,
        patch_col * PATCH:(patch_col + 1) * PATCH,
    ].T.reshape(16)
    assert np.array_equal(observed, expected)
del sample

retained_total = train_retained = test_retained = 0
started = time.perf_counter()
with DESCRIPTOR_BIN.open("wb") as destination:
    for start in range(0, images.shape[0], CHUNK_IMAGES):
        end = min(start + CHUNK_IMAGES, images.shape[0])
        chunk = images[start:end]
        blocks = chunk.reshape(-1, GRID, PATCH, GRID, PATCH).transpose(0, 1, 3, 2, 4)
        keep = np.any(blocks > THRESHOLD, axis=(-2, -1))
        vectors = blocks.swapaxes(-1, -2).reshape(-1, 16)
        selected = vectors[keep.reshape(-1)]
        descriptors = selected.astype(np.float32)
        descriptors /= np.float32(255.0)
        descriptors.tofile(destination)
        retained_total += int(keep.sum())
        train_in_chunk = max(0, min(end, CONFIG["train_images"]) - start)
        if train_in_chunk:
            train_retained += int(keep[:train_in_chunk].sum())
        test_offset = max(0, CONFIG["train_images"] - start)
        if end > CONFIG["train_images"]:
            test_retained += int(keep[test_offset:].sum())
        del chunk, blocks, keep, vectors, selected, descriptors

del images
gc.collect()
if retained_total != CONFIG["expected_descriptors"]:
    raise ValueError(
        f"Expected {CONFIG['expected_descriptors']} descriptors, got {retained_total}"
    )
assert DESCRIPTOR_BIN.stat().st_size == retained_total * 16 * 4
DESCRIPTOR_SHA256 = sha256_file(DESCRIPTOR_BIN)
print("Train descriptors:", train_retained, "| test descriptors:", test_retained)
print("Total descriptors:", retained_total, "| Julia matrix: 16 ×", retained_total)
print("Descriptor SHA-256:", DESCRIPTOR_SHA256)
print(f"Extraction seconds: {time.perf_counter() - started:.1f}")


## 5. Fit đúng một codebook TC4

Julia nạp ma trận 16×N và gọi Clustering.jl với K-means++, seed 123, squared Euclidean, maxiter 300 và tolerance 1e-4. Kết quả lưu vòng lặp thực tế, trạng thái hội tụ, distortion, occupancy, entropy, thời gian và checksum.


In [ ]:
JULIA_RUNNER = WORKDIR / "run_tc4_kmeans.jl"
JULIA_RUNNER.write_text(r'''
using Clustering, Distances, Random, SHA, Statistics, TOML
const K = parse(Int, ENV["FASHION_TC4_K"])
const SEED = parse(Int, ENV["FASHION_TC4_SEED"])
const MAXITER = parse(Int, ENV["FASHION_TC4_MAXITER"])
const TOL = parse(Float64, ENV["FASHION_TC4_TOL"])
const N = parse(Int, ENV["FASHION_TC4_N"])
const XPATH = ENV["FASHION_TC4_X"]
const INPUT_PATH = ENV["FASHION_TC4_INPUT"]
const OUT = ENV["FASHION_TC4_OUT"]

filesize(XPATH) == 16 * N * sizeof(Float32) || error("Wrong descriptor file size")
X = Matrix{Float32}(undef, 16, N)
open(XPATH, "r") do io
    read!(io, X)
    eof(io) || error("Unexpected trailing descriptor bytes")
end
all(isfinite, X) || error("Non-finite descriptor")
all(x -> 0f0 <= x <= 1f0, X) || error("Descriptor outside [0,1]")

println("Julia ", VERSION, " | Clustering.jl ", pkgversion(Clustering))
println("Input: ", size(X), " Float32; K=", K, "; seed=", SEED)
started = time()
model = Clustering.kmeans(
    X, K; init=:kmpp, maxiter=MAXITER, tol=TOL, display=:none,
    distance=SqEuclidean(), rng=MersenneTwister(SEED),
)
elapsed = time() - started
counts = [count(==(i), model.assignments) for i in 1:K]
p = counts ./ N
p = filter(>(0), p)
entropy = -sum(p .* log.(p)) / log(K)

centroids_path = joinpath(OUT, "centroids.csv")
open(centroids_path, "w") do io
    println(io, "cluster_id,", join(["pixel_" * lpad(string(i), 2, '0') for i in 1:16], ","))
    for cluster in 1:K
        println(io, cluster, ",", join(string.(model.centers[:, cluster]), ","))
    end
end
input = TOML.parsefile(INPUT_PATH)
summary = Dict{String,Any}(
    "status" => (model.converged ? "FITTED" : "NON_CONVERGED"),
    "stop_reason" => (model.converged ? "library_tolerance" : "max_iterations"),
    "dataset" => "Fashion-MNIST",
    "images_in_fit_scope" => 70000,
    "train_images_in_fit_scope" => 60000,
    "test_images_in_fit_scope" => 10000,
    "labels_used_for_fit" => false,
    "descriptor_count" => N,
    "descriptor_dimensions" => 16,
    "descriptor_dtype" => "Float32",
    "descriptor_sha256" => bytes2hex(open(sha256, XPATH)),
    "archive_sha256" => input["archive_sha256"],
    "k" => K,
    "seed" => SEED,
    "initialization" => "kmeans++",
    "distance" => "squared_euclidean",
    "max_iterations" => MAXITER,
    "tolerance" => TOL,
    "iterations" => model.iterations,
    "converged" => model.converged,
    "total_cost" => Float64(model.totalcost),
    "distortion_mean_squared" => Float64(model.totalcost / N),
    "cluster_occupancy_used" => count(>(0), counts),
    "cluster_counts" => counts,
    "top_cluster_share" => maximum(counts) / N,
    "normalized_entropy" => Float64(entropy),
    "fit_seconds" => elapsed,
    "backend" => "CPU",
    "julia_version" => string(VERSION),
    "julia_threads" => Threads.nthreads(),
    "clustering_library" => "Clustering.jl",
    "clustering_version" => string(pkgversion(Clustering)),
    "centroids_sha256" => bytes2hex(open(sha256, centroids_path)),
)
summary_path = joinpath(OUT, "run_summary.toml")
open(summary_path, "w") do io
    TOML.print(io, summary)
end
println("RESULT status=", summary["status"],
    " | iterations=", model.iterations, "/", MAXITER,
    " | converged=", model.converged,
    " | distortion=", summary["distortion_mean_squared"],
    " | occupied=", summary["cluster_occupancy_used"], "/", K,
    " | fit_seconds=", round(elapsed; digits=2))
println("Summary: ", summary_path)
println("Centroids: ", centroids_path)
''')

env = JULIA_ENVIRONMENT.copy()
env.update({
    "FASHION_TC4_K": str(CONFIG["k"]),
    "FASHION_TC4_SEED": str(CONFIG["seed"]),
    "FASHION_TC4_MAXITER": str(CONFIG["max_iterations"]),
    "FASHION_TC4_TOL": str(CONFIG["tolerance"]),
    "FASHION_TC4_N": str(retained_total),
    "FASHION_TC4_X": str(DESCRIPTOR_BIN),
    "FASHION_TC4_INPUT": str(INPUT_MANIFEST),
    "FASHION_TC4_OUT": str(OUTPUT_DIR),
})
subprocess.run(
    [JULIA_BIN, "--startup-file=no", f"--project={JULIA_ENV_DIR}", str(JULIA_RUNNER)],
    cwd=WORKDIR, env=env, check=True,
)


## 6. Đọc kết quả

Bảng in ra số vòng mà Clustering.jl trả về. Summary TOML chứa cấu hình, trạng thái và checksum; centroids CSV có 32 hàng, mỗi hàng là một centroid 16 chiều. Tải hai file trong thư mục output ở panel Files của Colab.


In [ ]:
import tomllib

SUMMARY_PATH = OUTPUT_DIR / "run_summary.toml"
CENTROIDS_PATH = OUTPUT_DIR / "centroids.csv"
summary = tomllib.loads(SUMMARY_PATH.read_text())
print("Trạng thái:", summary["status"])
print("Số vòng K-means:", f"{summary['iterations']} / {summary['max_iterations']}")
print("Hội tụ:", summary["converged"])
print("Thời gian fit (giây):", round(summary["fit_seconds"], 2))
print("Distortion trung bình bình phương:", summary["distortion_mean_squared"])
print("Cụm có phần tử:", f"{summary['cluster_occupancy_used']} / {summary['k']}")
print("Entropy chuẩn hóa:", summary["normalized_entropy"])
print("Julia:", summary["julia_version"])
print("Clustering.jl:", summary["clustering_version"])
print("Descriptor SHA-256:", summary["descriptor_sha256"])
print("Summary:", SUMMARY_PATH)
print("Centroids:", CENTROIDS_PATH)
if not summary["converged"]:
    print("Chạm trần vòng lặp mà chưa hội tụ; giữ nguyên cấu hình và ghi NON_CONVERGED.")
